In [ ]:
from pynq import Overlay
from pynq import DefaultIP
from pynq import MMIO
from pynq import allocate
import numpy as np
import struct

BRAM_ADDR = 0xA0010000
ADDRESS_RANGE = 0x1000
ADDRESS_OFFSET = 0x00

class AddDriver(DefaultIP):

    bindto = ['xilinx.com:hls:firCoefficients:1.0']

    def __init__(self, description):
        super().__init__(description)

    def process(self, lowerCutoff, upperCutoff, samplingRate):
        self.write(0x10, lowerCutoff)
        self.write(0x18, upperCutoff)
        self.write(0x20, samplingRate)
        self.write(0x28, BRAM_ADDR)



overlay = Overlay('Motor_Simulator.bit')   
add_ip = overlay.firCoefficients
dma = overlay.axi_dma
dma_send = overlay.axi_dma.sendchannel
dma_recv = overlay.axi_dma.recvchannel



def writeCoefficients(lowerCutoff, upperCutoff, samplingRate):
        overlay.firCoefficients.process(lowerCutoff, upperCutoff, samplingRate)

def axi_stream(integerList):
    data_size = 100
    input_buffer = allocate(shape=(data_size,), dtype=np.uint32) 
    for i in range(len(integerList)): 
        input_buffer[i] = integerList[i]
    dma_send.transfer(input_buffer) 
    output_buffer = allocate(shape=(data_size,), dtype=np.uint32) 
    dma_recv.transfer(output_buffer)
    for i in range(len(integerList)): 
        integerList[i] = output_buffer[i] 
    del input_buffer, output_buffer 
    return integerList









In [ ]:
writeCoefficients(500, 10000, 20000)
bram = MMIO(BRAM_ADDR, ADDRESS_RANGE)

#bram.write(0x20, 1)
result = bram.read(0x4 * 0)
print(result)

In [ ]:

print(axi_stream([2,2]))
